# 11 · Spark + RustFS via `s3a://` (Case D)

**Theory**: docs/08-spark-and-object-storage.md

**Prerequisite**: `make up-s3` (starts the Standalone cluster from Case B
*plus* a single-node RustFS server with 4 drives / Erasure Coding RS(4,2) —
the same pattern as `cdn-s3-lab`).

The `spark-connect` container's startup command in `docker-compose.yml`
already carries the `spark.hadoop.fs.s3a.*` configuration pointing at
`rustfs-server:9000` — your thin client doesn't need to repeat it.

In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, layer_path

spark = get_connect_session("11-spark-s3-rustfs")
spark

## Bootstrapping Bronze on S3

`make up-s3` already created empty `bronze`/`silver`/`gold` buckets
(`rustfs-init` in `docker-compose.yml`). We write the same locally
-generated dataset into `s3a://bronze/...` from the container side (which
has `/data` mounted, so it can see the host-generated Parquet files) — no
separate upload tool needed here, unlike Case C.

In [ ]:
local_vendas = spark.read.parquet(layer_path("connect", "bronze", "vendas"))
local_empresas = spark.read.parquet(layer_path("connect", "bronze", "empresas"))

local_vendas.write.mode("overwrite").parquet(layer_path("s3", "bronze", "vendas"))
local_empresas.write.mode("overwrite").parquet(layer_path("s3", "bronze", "empresas"))
print("Bronze layer written to s3a://bronze/")

In [ ]:
vendas_s3 = spark.read.parquet(layer_path("s3", "bronze", "vendas"))
print(f"Read back from RustFS: {vendas_s3.count():,} rows")
vendas_s3.show(5)

## Partition pruning: why it matters even more on S3

Since S3 has no data locality (docs/08), skipping irrelevant partitions
before any bytes cross the network matters more here than on HDFS. Compare
a full scan against a filtered read on `ano`/`mes` and check `.explain()`
for a `PartitionFilters` entry.

In [ ]:
import time

from pyspark.sql.functions import sum as spark_sum

start = time.perf_counter()
full = vendas_s3.agg(spark_sum("valor")).collect()
full_seconds = time.perf_counter() - start

filtered = vendas_s3.filter("ano = 2026 AND mes = 7")
filtered.explain()  # look for PartitionFilters in the plan

start = time.perf_counter()
filtered_result = filtered.agg(spark_sum("valor")).collect()
filtered_seconds = time.perf_counter() - start

print(f"Full scan:     {full_seconds:.2f}s")
print(f"Filtered scan: {filtered_seconds:.2f}s (only ano=2026/mes=7 partitions read)")

## Writing Silver and Gold

Same medallion pattern as every other case — only the storage layer changed.

In [ ]:
from pyspark.sql.functions import broadcast

empresas_s3 = spark.read.parquet(layer_path("s3", "bronze", "empresas"))

silver = vendas_s3.filter("valor > 0")
silver.write.mode("overwrite").parquet(layer_path("s3", "silver", "vendas"))

gold = (
    spark.read.parquet(layer_path("s3", "silver", "vendas"))
    .join(broadcast(empresas_s3), "id_empresa")
    .groupBy("setor", "ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
)
gold.write.mode("overwrite").parquet(layer_path("s3", "gold", "vendas_por_setor"))
print("Gold layer written. Check the RustFS console: http://localhost:9001")

In [ ]:
spark.stop()